# Prepare WeavePop run of Desjardins dataset

In [4]:
# Select Kernel: python_env
import os
import pandas as pd
os.chdir("/FastData/czirion/WeavePop_Cneoformans")

Paths:

In [ ]:
path_chromosomes = "/FastData/czirion/cryptococcus_reference_genomes/chromosomes.csv"
output_chromosomes = "Crypto_Desjardins/config/chromosomes.csv"
path_data_refs = "data/references/"
path_data_samples = "data/samples/"

## Gather chromosomes table

In [ ]:
chromosomes = pd.read_csv(path_chromosomes, header = 0)
# select only lineages VNI, VNBI, VNBII, VNII
chromosomes = chromosomes[chromosomes['lineage'].isin(['VNI', 'VNBI', 'VNBII', 'VNII'])]
# select only chromosomes chr01 to chr14
chromosomes = chromosomes[chromosomes['chromosome'].isin([f'chr{str(i).zfill(2)}' for i in range(1, 15)])]
chromosomes.to_csv(output_chromosomes, index = False)

## Gather reference genomes

In [19]:
%%bash
mkdir -p Crypto_Desjardins/data/references/
for lineage in VNI VNII VNBI VNBII
do
    grep -w $lineage /FastData/czirion/cryptococcus_reference_genomes/reference_genomes.csv | while read line
    do
        sp=$(echo $line | cut -d"," -f1)
        ln=$(echo $line | cut -d"," -f2)
        st=$(echo $line | cut -d"," -f3)
        file="/FastData/czirion/cryptococcus_reference_genomes/alignments_to_H99/${sp}_${ln}_${st}_reoriented.fasta"
        scp $file Crypto_Desjardins/data/references/${ln}.fasta
    done
done


Gather VNI GFF

In [22]:
%%bash
grep -w VNI /FastData/czirion/cryptococcus_reference_genomes/reference_genomes.csv | while read line
do
    sp=$(echo $line | cut -d"," -f1)
    ln=$(echo $line | cut -d"," -f2)
    st=$(echo $line | cut -d"," -f3)
    file="/FastData/czirion/cryptococcus_reference_genomes/alignments_to_H99/${sp}_${ln}_${st}_reoriented.gff"
    scp $file Crypto_Desjardins/data/references/${ln}.gff
done

### Remove mitochondria from VNI

In an environment with seqkit
```
seqkit grep -n -r -v -p CP003834.1 Crypto_Desjardins/data/references/VNI.fasta > Crypto_Desjardins/data/references/VNI.fasta.modif
mv Crypto_Desjardins/data/references/VNI.fasta Crypto_Desjardins/data/references/VNI.fasta.original
mv Crypto_Desjardins/data/references/VNI.fasta.modif Crypto_Desjardins/data/references/VNI.fasta

grep -v CP003834.1 Crypto_Desjardins/data/references/VNI.gff > Crypto_Desjardins/data/references/VNI.gff.modif
mv Crypto_Desjardins/data/references/VNI.gff Crypto_Desjardins/data/references/VNI.gff.original
mv Crypto_Desjardins/data/references/VNI.gff.modif Crypto_Desjardins/data/references/VNI.gff
```


## Gather metadata

Gather the standardized Desjardins metadata table which already has the VNIa sublineages included from the Ashton metadata

In [24]:
%%bash 
scp /BigData/czirion/data_public/ashton/metadata_desjardins.csv Crypto_Desjardins/config/metadata.csv

In [5]:
%%bash 
scp /BigData/czirion/data_public/desjardins/download_files/files/runs_table.tsv Crypto_Desjardins/config/

## Gather the cleaned FASTQs

In [28]:
%%bash
ln -s /BigData/czirion/data_public/desjardins/data_clean Crypto_Desjardins/data/samples

Download the RepBase

In [ ]:
%%bash
wget https://www.girinst.org/server/RepBase/protected/RepBase30.11.fasta.tar.gz

In [ ]:
%%bash
tar -xvzf Crypto_Desjardins/config/RepBase30.11.fasta.tar.gz
cat RepBase30.11.fasta/*.ref > Crypto_Desjardins/config/RepBase.fasta
cat RepBase30.11.fasta/appendix/*.ref >> Crypto_Desjardins/config/RepBase.fasta
rm -rf RepBase30.11.fasta/ Crypto_Desjardins/config/RepBase30.11.fasta.tar.gz

## Create loci table